# MedCLIP-SAMv2 Text+Boxes — Sheffield Dataset — GPU Optimised

Identical pipeline to `lambda_medclipsamv2_textboxes_sheffield.ipynb` with two
GPU optimisations:

| Change | Original | Optimised |
|---|---|---|
| SAM encoder | batch\_size=1 per slice | auto-batched (EMBED\_BATCH, VRAM-scaled) |
| BiomedCLIP saliency | BATCH=64 (hardcoded) | SAL\_BATCH (configurable) |

`EMBED_BATCH` is auto-detected from available VRAM at startup:
ViT-B needs ~1.5 GB activations per 1024² image → `(vram_gb - 2) / 1.5`, capped at 32.
On a 24 GB A10 this gives ~14; on an 80 GB A100/H100 it gives 32.

⚠️ **Requires MuscleMap WB Sheffield segmentations** — run
`lambda_musclemap_wb_sheffield.ipynb` first and download results to
`eval_notebooks/muscle_map_wb/sheffield_segs/` before uploading here.

Data: `~/sheffeld/20440164/Aug_N.dcm`
MM WB segs: `~/musclemap_wb_sheffield_segs/Aug_N_dseg.nii.gz`
Output: `~/medclipsamv2_textboxes_sheffield_segs/Aug_N_mcsam2textboxes.npz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/sheffeld \
  ubuntu@<YOUR-LAMBDA-IP>:~/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/muscle_map_wb/sheffield_segs/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/musclemap_wb_sheffield_segs/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  "/tmp/docker-desktop-root/run/desktop/mnt/host/c/Users/docto/AppData/Local/Dafne-imaging/Dafne/models/medsam_vit_b.pth" \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsam_vit_b.pth
```

## 2 — Download results when done
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/medclipsamv2_textboxes_sheffield_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medclipsamv2textboxes/sheffield_segs/
```
**Terminate the instance when done.**

In [ ]:
import subprocess, sys

# Reinstall PyTorch with a wheel compiled against NumPy 2.x
# (the system torch is built for NumPy 1.x, which conflicts with opencv>=4.10)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu124'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/facebookresearch/segment-anything.git'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'SimpleITK', 'scikit-image', 'pydicom', 'opencv-python-headless'])

import importlib, torch
importlib.invalidate_caches()
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import os

REPO_DIR = os.path.expanduser('~/MedCLIP-SAMv2')
VENV_DIR = os.path.expanduser('~/mcsam2_env')
VENV_PY  = os.path.join(VENV_DIR, 'bin', 'python')

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone',
        'https://github.com/HealthX-Lab/MedCLIP-SAMv2.git', REPO_DIR])

if not os.path.exists(VENV_PY):
    subprocess.check_call([sys.executable, '-m', 'venv', VENV_DIR])

def venv_pip(*args):
    subprocess.check_call([VENV_PY, '-m', 'pip'] + list(args))

venv_pip('install', '-q', '--upgrade', 'pip')
venv_pip('install', '-q', '--upgrade',
    'torch', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu124')
venv_pip('install', '-q', '-e', os.path.join(REPO_DIR, 'segment-anything'))
venv_pip('install', '-q', 'git+https://github.com/lucasb-eyer/pydensecrf.git')
# No opencv-python here — it requires numpy>=2 and would break open_clip/grad-cam.
# The batch saliency script uses PIL for image I/O instead.
venv_pip('install', '-q',
    'open_clip_torch', 'SimpleITK', 'Pillow',
    'huggingface_hub', 'transformers<4.46',
    'matplotlib', 'grad-cam', 'pandas', 'tqdm', 'scipy', 'scikit-learn')
# Pin numpy<2 LAST so no earlier install can upgrade it
venv_pip('install', '-q', 'numpy<2')
import subprocess as _sp
_np_ver = _sp.check_output([VENV_PY, '-c', 'import numpy; print(numpy.__version__)'], text=True).strip()
print(f'Venv numpy: {_np_ver}')
assert _np_ver.startswith('1.'), f'Expected numpy 1.x in venv, got {_np_ver}'
print('Venv dependencies installed.')

In [ ]:
import glob, re, shutil, tempfile
import numpy as np
import SimpleITK as sitk
import torch
import torch.nn.functional as F
import cv2
import pydicom
from PIL import Image
from skimage import transform
from segment_anything import sam_model_registry

IMG_DIR    = os.path.expanduser('~/sheffeld/20440164')
MM_SEG_DIR = os.path.expanduser('~/musclemap_wb_sheffield_segs')
OUTPUT_DIR = os.path.expanduser('~/medclipsamv2_textboxes_sheffield_segs')
MEDSAM_CKPT= os.path.expanduser('~/medsam_vit_b.pth')
REPO_DIR   = os.path.expanduser('~/MedCLIP-SAMv2')
VENV_PY    = os.path.expanduser('~/mcsam2_env/bin/python')
VENV_DIR   = os.path.expanduser('~/mcsam2_env')
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
SAM_DEVICE = DEVICE

os.makedirs(OUTPUT_DIR, exist_ok=True)

for label, path in [('MedSAM ckpt', MEDSAM_CKPT), ('MM WB segs', MM_SEG_DIR)]:
    ok = os.path.exists(path)
    print(f'  {"OK" if ok else "MISSING"}: {label}')
    if not ok:
        raise FileNotFoundError(f'{label} not found — upload it first')

dcm_files = sorted(
    [f for f in glob.glob(os.path.join(IMG_DIR, 'Aug_*.dcm'))
     if '_segmentations' not in f],
    key=lambda p: int(re.search(r'Aug_(\d+)\.dcm', p).group(1)),
)

# ── GPU batch sizes ───────────────────────────────────────────────────────────
def _auto_embed_batch(device):
    """SAM ViT-B needs ~1.5 GB VRAM per image at 1024². Scale to available memory."""
    if device == 'cpu':
        return 1
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    return max(1, min(32, int((vram_gb - 2) / 1.5)))

EMBED_BATCH = _auto_embed_batch(SAM_DEVICE)  # SAM image encoder slices per forward pass
SAL_BATCH   = 128                            # BiomedCLIP saliency slices per forward pass

print(f'{len(dcm_files)} DICOM volumes')
print(f'SAM device  : {SAM_DEVICE}')
print(f'EMBED_BATCH : {EMBED_BATCH}  (SAM encoder slices per forward pass)')
print(f'SAL_BATCH   : {SAL_BATCH}  (BiomedCLIP slices per forward pass)')

In [ ]:
# (output_key, text_prompt, mm_wb_label)
# MuscleMap WB uses 7xxx label scheme
MUSCLES = [
    ('L_vastus_lateralis',    'vastus lateralis muscle left thigh MRI axial cross section',   7101),
    ('R_vastus_lateralis',    'vastus lateralis muscle right thigh MRI axial cross section',  7102),
    ('L_vastus_intermedius',  'vastus intermedius muscle left thigh MRI axial cross section', 7111),
    ('R_vastus_intermedius',  'vastus intermedius muscle right thigh MRI axial cross section',7112),
    ('L_vastus_medialis',     'vastus medialis muscle left thigh MRI axial cross section',    7121),
    ('R_vastus_medialis',     'vastus medialis muscle right thigh MRI axial cross section',   7122),
    ('L_rectus_femoris',      'rectus femoris muscle left thigh MRI axial cross section',     7131),
    ('R_rectus_femoris',      'rectus femoris muscle right thigh MRI axial cross section',    7132),
    ('L_sartorius',           'sartorius muscle left thigh MRI axial cross section',          7141),
    ('R_sartorius',           'sartorius muscle right thigh MRI axial cross section',         7142),
    ('L_gracilis',            'gracilis muscle left thigh MRI axial cross section',           7151),
    ('R_gracilis',            'gracilis muscle right thigh MRI axial cross section',          7152),
    ('L_semimembranosus',     'semimembranosus muscle left thigh MRI axial cross section',    7161),
    ('R_semimembranosus',     'semimembranosus muscle right thigh MRI axial cross section',   7162),
    ('L_semitendinosus',      'semitendinosus muscle left thigh MRI axial cross section',     7171),
    ('R_semitendinosus',      'semitendinosus muscle right thigh MRI axial cross section',    7172),
    ('L_biceps_femoris_long', 'biceps femoris long head muscle left thigh MRI axial cross section',  7181),
    ('R_biceps_femoris_long', 'biceps femoris long head muscle right thigh MRI axial cross section', 7182),
    ('L_biceps_femoris_short','biceps femoris short head muscle left thigh MRI axial cross section', 7191),
    ('R_biceps_femoris_short','biceps femoris short head muscle right thigh MRI axial cross section',7192),
    ('L_adductor_magnus',     'adductor magnus muscle left thigh MRI axial cross section',    7201),
    ('R_adductor_magnus',     'adductor magnus muscle right thigh MRI axial cross section',   7202),
    ('L_adductor_longus',     'adductor longus muscle left thigh MRI axial cross section',    7211),
    ('R_adductor_longus',     'adductor longus muscle right thigh MRI axial cross section',   7212),
    ('L_adductor_brevis',     'adductor brevis muscle left thigh MRI axial cross section',    7221),
    ('R_adductor_brevis',     'adductor brevis muscle right thigh MRI axial cross section',   7222),
]
print(f'{len(MUSCLES)} muscles')

In [ ]:
import glob as _glob, json as _json
_venv_site = _glob.glob(os.path.join(VENV_DIR, 'lib', 'python3.*', 'site-packages'))
VENV_SITE  = _venv_site[0] if _venv_site else ''

SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONPATH']       = VENV_SITE + ':' + SUBPROCESS_ENV.get('PYTHONPATH', '')
SUBPROCESS_ENV['PYTHONNOUSERSITE'] = '1'
SUBPROCESS_ENV['MPLBACKEND']       = 'Agg'

BATCH_SAL_SCRIPT = os.path.expanduser('~/batch_saliency.py')

_SAL_SCRIPT_BODY = '''#!/usr/bin/env python3
"""Batch BiomedCLIP saliency — one model load per volume, GPU-batched inference.
Images are kept on CPU and moved to GPU per-batch to minimise VRAM footprint."""
import argparse, json, os
import numpy as np, torch
from PIL import Image
import open_clip

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument(\'--png-dir\',      required=True)
    p.add_argument(\'--out-base\',     required=True)
    p.add_argument(\'--prompts-json\', required=True)
    p.add_argument(\'--device\',       default=\'cuda\')
    p.add_argument(\'--batch-size\',   type=int, default=64)
    return p.parse_args()

def main():
    args   = parse_args()
    device = args.device if torch.cuda.is_available() else \'cpu\'

    with open(args.prompts_json) as f:
        prompts = json.load(f)

    png_files = sorted(
        [x for x in os.listdir(args.png_dir) if x.endswith(\'.png\')],
        key=lambda x: int(x.split(\'.\')[0])
    )
    N = len(png_files)
    print(f\'Loading BiomedCLIP on {device} ...\', flush=True)
    model, _, preprocess = open_clip.create_model_and_transforms(
        \'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224\'
    )
    tokenizer = open_clip.get_tokenizer(
        \'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224\'
    )
    model = model.to(device).eval()

    # Auto-detect safe batch size from free VRAM after model load.
    # ViT-B/16 needs ~6 MB per image (activations + gradients during backward).
    BATCH = args.batch_size
    if str(device) != \'cpu\' and torch.cuda.is_available():
        _free_gb = torch.cuda.mem_get_info()[0] / 1e9
        BATCH    = max(4, min(args.batch_size, int(_free_gb / 0.006)))
        print(f\'[sal] free VRAM after model load: {_free_gb:.1f} GB  SAL_BATCH={BATCH}\', flush=True)

    print(f\'Model ready. {len(prompts)} prompts x {N} slices. batch={BATCH}\', flush=True)

    imgs_pil       = [Image.open(os.path.join(args.png_dir, f)).convert(\'RGB\') for f in png_files]
    W_orig, H_orig = imgs_pil[0].size
    # Keep preprocessed images on CPU — moved to GPU per-batch to avoid VRAM pressure
    img_batch_cpu  = torch.stack([preprocess(im) for im in imgs_pil])

    for name, prompt in prompts.items():
        out_dir = os.path.join(args.out_base, name)
        os.makedirs(out_dir, exist_ok=True)
        print(f\'  [{name}]\', flush=True)

        text_tok = tokenizer([prompt]).to(device)
        with torch.no_grad():
            text_feat = model.encode_text(text_tok)
            text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

        sal_chunks = []
        for b0 in range(0, N, BATCH):
            b1   = min(b0 + BATCH, N)
            imgs = img_batch_cpu[b0:b1].to(device).detach().requires_grad_(True)
            img_feat = model.encode_image(imgs)
            img_feat = img_feat / (img_feat.norm(dim=-1, keepdim=True) + 1e-8)
            sim = (img_feat * text_feat).sum(dim=-1).sum()
            sim.backward()
            sal = imgs.grad.detach().abs().max(dim=1)[0].cpu().numpy()
            sal_chunks.append(sal)
            del imgs
            torch.cuda.empty_cache()

        sal_all = np.concatenate(sal_chunks, axis=0)
        s_min, s_max = sal_all.min(), sal_all.max()
        if s_max > s_min:
            sal_all = (sal_all - s_min) / (s_max - s_min)

        for i, fn in enumerate(png_files):
            sal_u8 = (sal_all[i] * 255).astype(np.uint8)
            Image.fromarray(sal_u8).resize(
                (W_orig, H_orig), Image.BILINEAR
            ).save(os.path.join(out_dir, fn))

    print(\'Batch saliency complete.\', flush=True)

if __name__ == \'__main__\':
    main()
'''

with open(BATCH_SAL_SCRIPT, 'w') as _f:
    _f.write(_SAL_SCRIPT_BODY)
print(f'Batch saliency script written to {BATCH_SAL_SCRIPT}')


# ── numpy <-> torch bridges that survive has_numpy=False ─────────────────────

def _np_to_t(arr: np.ndarray) -> torch.Tensor:
    a = np.ascontiguousarray(arr, dtype=np.float32)
    return torch.frombuffer(a, dtype=torch.float32).clone().reshape(a.shape)

def _t_to_np(t: torch.Tensor) -> np.ndarray:
    t_con = t.detach().cpu().float().clone().contiguous()
    raw   = bytes(t_con.untyped_storage())
    return np.frombuffer(raw, dtype=np.float32).reshape(t_con.shape).copy()


# ── Helpers ───────────────────────────────────────────────────────────────────

def preprocess_slice(sl_arr: np.ndarray) -> torch.Tensor:
    """(H, W) float32 → (3, 1024, 1024) float32 CPU tensor ready for SAM encoder."""
    img_norm = (sl_arr * 255.0 / (sl_arr.max() + 1e-8)).astype(np.float32)
    img_3c   = np.stack([img_norm, img_norm, img_norm], axis=-1)
    img_1024 = cv2.resize(img_3c, (1024, 1024), interpolation=cv2.INTER_CUBIC)
    lo, hi   = img_1024.min(), img_1024.max()
    img_1024 = ((img_1024 - lo) / max(hi - lo, 1e-8)).astype(np.float32)
    arr      = img_1024.transpose(2, 0, 1).copy()
    return torch.frombuffer(arr, dtype=torch.float32).clone().reshape(3, 1024, 1024)


def export_slices_as_png(img_array, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for i in range(img_array.shape[0]):
        sl = img_array[i]
        sl_norm = (sl - sl.min()) / (sl.max() - sl.min() + 1e-8)
        Image.fromarray(
            np.stack([(sl_norm * 255).astype(np.uint8)] * 3, axis=-1)
        ).save(os.path.join(out_dir, f'{i}.png'))


def run_all_saliencies(png_dir, out_base, muscles, sal_batch):
    """One subprocess: load BiomedCLIP once, run all muscle prompts."""
    prompts = {name: prompt for name, prompt, _ in muscles}
    prompts_path = os.path.join(out_base, '_prompts.json')
    os.makedirs(out_base, exist_ok=True)
    with open(prompts_path, 'w') as f:
        _json.dump(prompts, f)
    sal_env = {**SUBPROCESS_ENV, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
    r = subprocess.run(
        [VENV_PY, BATCH_SAL_SCRIPT,
         '--png-dir',      png_dir,
         '--out-base',     out_base,
         '--prompts-json', prompts_path,
         '--device',       DEVICE,
         '--batch-size',   str(sal_batch)],
        capture_output=True, text=True, env=sal_env,
    )
    if r.returncode != 0:
        raise RuntimeError(f'Batch saliency failed:\n{r.stderr[-3000:]}')
    print(r.stdout[-800:])


def load_saliency_volume(sal_dir, num_slices, target_size=256):
    vol = np.zeros((num_slices, target_size, target_size), dtype=np.float32)
    for i in range(num_slices):
        p = os.path.join(sal_dir, f'{i}.png')
        if os.path.exists(p):
            s = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
            if s is not None:
                s = cv2.resize(s, (target_size, target_size),
                               interpolation=cv2.INTER_LINEAR).astype(np.float32)
                vol[i] = (s / 255.0) * 12.0 - 6.0
    return _np_to_t(vol[:, None, :, :])


def get_mm_boxes(seg_array, mm_label, H, W, margin=5):
    boxes = []
    for sl in range(seg_array.shape[0]):
        mask = (seg_array[sl] == mm_label).astype(np.uint8)
        if not mask.any():
            boxes.append(None); continue
        rows = np.where(np.any(mask, axis=1))[0]
        cols = np.where(np.any(mask, axis=0))[0]
        r0, r1 = rows[[0, -1]]; c0, c1 = cols[[0, -1]]
        mH, mW = mask.shape
        box = np.array([
            max(0, c0-margin), max(0, r0-margin),
            min(mW-1, c1+margin), min(mH-1, r1+margin),
        ], dtype=float)
        boxes.append(box / np.array([W, H, W, H]) * 1024)
    return boxes


def medsam_infer(sam_model, img_embed, box_1024, saliency_256, H, W, device):
    box_t = torch.as_tensor(box_1024, dtype=torch.float, device=device)[None, None, :]
    sal_t = saliency_256.to(device)
    with torch.no_grad():
        sparse_emb, dense_emb = sam_model.prompt_encoder(
            points=None, boxes=box_t, masks=sal_t)
        logits, _ = sam_model.mask_decoder(
            image_embeddings=img_embed.to(device),
            image_pe=sam_model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_emb,
            dense_prompt_embeddings=dense_emb,
            multimask_output=False,
        )
    pred = F.interpolate(torch.sigmoid(logits), size=(H, W),
                         mode='bilinear', align_corners=False)
    return (_t_to_np(pred.squeeze()) > 0.5).astype(np.uint8)


print('Helpers defined.')

In [ ]:
sam_model = sam_model_registry['vit_b'](checkpoint=MEDSAM_CKPT)
sam_model.to(device=SAM_DEVICE).eval()
print('MedSAM loaded on', SAM_DEVICE)

In [ ]:
for dcm_path in dcm_files:
    idx      = re.search(r'Aug_(\d+)\.dcm', dcm_path).group(1)
    mm_path  = os.path.join(MM_SEG_DIR, f'Aug_{idx}_dseg.nii.gz')
    out_path = os.path.join(OUTPUT_DIR, f'Aug_{idx}_mcsam2textboxes.npz')

    if not os.path.exists(mm_path):
        print(f'[skip] no MM WB seg for Aug_{idx}')
        continue
    if os.path.exists(out_path):
        print(f'Skipping (done): Aug_{idx}')
        continue

    print(f'\n═══ Aug_{idx} ═══')
    ds        = pydicom.dcmread(dcm_path)
    img_array = ds.pixel_array.astype(np.float32)
    D, H, W   = img_array.shape

    seg_sitk  = sitk.ReadImage(mm_path)
    seg_array = sitk.GetArrayFromImage(seg_sitk).astype(np.int32)
    print(f'  Image: {img_array.shape}  MM seg: {seg_array.shape}')

    tmp_root  = tempfile.mkdtemp(prefix='mctbs_')
    all_masks = {}

    try:
        png_dir = os.path.join(tmp_root, 'slices')
        export_slices_as_png(img_array, png_dir)

        # ── Batched SAM encoder ───────────────────────────────────────────────
        print(f'  Computing SAM embeddings (EMBED_BATCH={EMBED_BATCH})...')
        embeddings = []
        for b0 in range(0, D, EMBED_BATCH):
            b1    = min(b0 + EMBED_BATCH, D)
            batch = torch.stack([preprocess_slice(img_array[sl]) for sl in range(b0, b1)])
            batch = batch.to(SAM_DEVICE)
            with torch.no_grad():
                embs = sam_model.image_encoder(batch)   # (B, 256, 64, 64)
            embeddings.extend([embs[i:i+1].cpu() for i in range(embs.shape[0])])
            del batch, embs
        torch.cuda.empty_cache()
        print(f'  {D} embeddings ready')

        # ── BiomedCLIP saliency ───────────────────────────────────────────────
        sal_base = os.path.join(tmp_root, 'saliencies')
        print(f'  Running batch saliency (SAL_BATCH={SAL_BATCH})...')
        run_all_saliencies(png_dir, sal_base, MUSCLES, SAL_BATCH)

        # ── Pre-load all saliency volumes and boxes ───────────────────────────
        # Single I/O phase before the decoder — eliminates per-muscle disk gaps.
        print(f'  Pre-loading {len(MUSCLES)} saliency volumes...')
        sal_vols   = []
        boxes_list = []
        for muscle_name, _, mm_label in MUSCLES:
            sal_dir = os.path.join(sal_base, muscle_name)
            sal_vols.append(load_saliency_volume(sal_dir, D))        # (D, 1, 256, 256)
            boxes_list.append(get_mm_boxes(seg_array, mm_label, H, W))  # D items

        # ── Per-slice decoder: all muscles in one forward pass ────────────────
        # SAM supports N prompts for one image in a single call.
        # Looping over slices (not muscles) batches all 26 muscle prompts per
        # slice → D calls instead of 26×D (26× fewer decoder launches).
        all_masks = {name: np.zeros((D, H, W), dtype=np.uint8) for name, _, _ in MUSCLES}
        print(f'  Decoding masks ({D} slices × {len(MUSCLES)} muscles/call)...')

        for sl_idx in range(D):
            valid_m   = [(m, boxes_list[m][sl_idx]) for m in range(len(MUSCLES))
                         if boxes_list[m][sl_idx] is not None]
            invalid_m = [m for m in range(len(MUSCLES))
                         if boxes_list[m][sl_idx] is None]

            # No-box slices: threshold saliency directly (no GPU needed)
            for m in invalid_m:
                sal_np = _t_to_np(sal_vols[m][sl_idx, 0])
                all_masks[MUSCLES[m][0]][sl_idx] = cv2.resize(
                    (sal_np > 0).astype(np.uint8), (W, H),
                    interpolation=cv2.INTER_NEAREST)

            if not valid_m:
                continue

            m_idxs  = [v[0] for v in valid_m]
            N       = len(valid_m)

            # Stack all N muscle prompts for this slice
            boxes_t = torch.tensor(
                [v[1] for v in valid_m], dtype=torch.float, device=SAM_DEVICE
            ).unsqueeze(1)                                            # (N, 1, 4)
            sals_t  = torch.cat(
                [sal_vols[m][sl_idx:sl_idx+1] for m in m_idxs]
            ).to(SAM_DEVICE)                                          # (N, 1, 256, 256)

            with torch.no_grad():
                sparse_emb, dense_emb = sam_model.prompt_encoder(
                    points=None, boxes=boxes_t, masks=sals_t)
                # image_embeddings=(1,...): SAM repeats it N times internally
                logits, _ = sam_model.mask_decoder(
                    image_embeddings=embeddings[sl_idx].to(SAM_DEVICE),
                    image_pe=sam_model.prompt_encoder.get_dense_pe(),
                    sparse_prompt_embeddings=sparse_emb,   # (N, 2, 256)
                    dense_prompt_embeddings=dense_emb,     # (N, 256, 64, 64)
                    multimask_output=False,
                )
            # logits: (N, 1, 64, 64) → upsample → threshold
            pred     = F.interpolate(torch.sigmoid(logits), size=(H, W),
                                     mode='bilinear', align_corners=False)
            masks_np = (_t_to_np(pred.squeeze(1)) > 0.5).astype(np.uint8)  # (N, H, W)

            for j, m in enumerate(m_idxs):
                all_masks[MUSCLES[m][0]][sl_idx] = masks_np[j]

        for name, _, _ in MUSCLES:
            print(f'  [{name}] {int(all_masks[name].sum()):,} voxels')

        np.savez_compressed(out_path, **all_masks)
        print(f'  Saved → {out_path}')

    except Exception as exc:
        import traceback
        print(f'  ERROR on Aug_{idx}: {exc}')
        traceback.print_exc()

    finally:
        shutil.rmtree(tmp_root, ignore_errors=True)

print('\nAll done.')

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.npz')))
print(f'Output files: {len(results)} / {len(dcm_files)}')
if results:
    s = np.load(results[0])
    for k in sorted(s.files):
        print(f'  {k}: voxels={int(s[k].sum()):,}')